# TiTiler-CMR Compatibility Report

## What datasets are compatible with TiTiler-CMR?

This report summarizes the 2026-07-01 compatibility assessment for TiTiler-CMR tiling endpoints. It distinguishes three outcomes:

- **Compatible:** the assessment found a tileable asset.
- **Incompatible:** the assessment found a concrete data, asset, access, render, request, or service failure.
- **Inconclusive:** the assessment could not make a compatibility determination. These datasets are probably incompatible with TiTiler-CMR as currently configured, but they may be worth exploring further when the blocker is operational, sampling-related, or caused by incomplete metadata.

The current assessment uses standard access paths. Authentication and credential failures are therefore visible in this report and should not be compared directly with older reports that bypassed some access checks.

This report provides:

1. A searchable listing of NASA EOSDIS datasets and their tiling compatibility outcome.
2. An overview of the dataset and failure categories observed in the assessment.

See [titiler-cmr-compatibility's METHODOLOGY](https://github.com/developmentseed/titiler-cmr-compatibility/blob/main/METHODOLOGY.md) documentation for a detailed explanation of how datasets were tested.



## What datasets are compatible?

In [ ]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import Markdown, display

notebook_dir = Path("docs/compatibility")
if notebook_dir.exists():
    sys.path.append(str(notebook_dir))
else:
    notebook_dir = Path.cwd()

plotting = importlib.import_module("plotting")
try:
    reporting = importlib.import_module("docs.compatibility.reporting")
except ModuleNotFoundError:
    reporting = importlib.import_module("reporting")

assessment_path = Path("assessment-results-2026-07-01.parquet")
if not assessment_path.exists():
    assessment_path = notebook_dir / assessment_path

df = reporting.classify_assessment(pd.read_parquet(assessment_path))
num_all_datasets = len(df)
compatible_datasets = df[df.tiling_compatible]
num_compatible_datasets = len(compatible_datasets)
status_counts = df["assessment_status"].value_counts()

display(
    Markdown(
        f"Out of {num_all_datasets} datasets tested, **{num_compatible_datasets} were found to be compatible**. "
        f"The assessment also found **{status_counts.get('incompatible', 0)} incompatible** datasets and "
        f"**{status_counts.get('inconclusive', 0)} inconclusive** datasets."
    )
)

### Searchable Dataset Table

To search across all datasets (where a granule was found) and check compatibility status, view the interactive table:

**[→ View Interactive Dataset Table](results_table.html)**

To determine if a particular dataset is compatible, search for it by name and check the `Tiling Compatible` column.

To regenerate the table, run the [`generate_table.py`](https://github.com/developmentseed/titiler-cmr/blob/develop/docs/compatibility/generate_table.py) script.

## What types of datasets are compatible?

Compatibility is concentrated in datasets whose assets can be opened by TiTiler-CMR's rasterio or xarray-backed readers and whose metadata identifies a tileable variable and spatial coordinates. The charts below summarize compatible datasets by data center, processing level, and file extension.



### Compatible Datasets by Data Center

In [ ]:
# Keep this dense pie readable by showing labels in the legend only.
data_center_counts = compatible_datasets.data_center.value_counts(dropna=False)
data_center_names = data_center_counts.index

plotting.create_pie_chart(
    labels=data_center_names,
    sizes=data_center_counts,
    title="Compatible Datasets by Data Center",
    legend=True,
    top_n=8,
)

### Percentage of compatible datasets by data center

In [ ]:
result = df.groupby("data_center")["tiling_compatible"].mean() * 100
result = result.map("{:.2f}%".format)
result.reset_index().rename(
    columns={
        "data_center": "Data Center",
        "tiling_compatible": "% Tiling-Compatible Datasets",
    }
)

### Compatible datasets by backend

This table shows which backend found a tileable asset for each compatible dataset.


In [ ]:
backend_counts = (
    compatible_datasets["backend"]
    .fillna("Unknown")
    .value_counts()
    .rename_axis("Backend")
    .reset_index(name="Compatible Datasets")
)
backend_counts["Share of Compatible Datasets"] = (
    backend_counts["Compatible Datasets"] / num_compatible_datasets * 100
).map("{:.1f}%".format)
backend_counts

### Compatible Datasets by Processing Level

In [ ]:
processing_level_counts = compatible_datasets.processing_level.value_counts(
    dropna=False
)
processing_level_ids = processing_level_counts.index

plotting.create_pie_chart(
    labels=processing_level_ids,
    sizes=processing_level_counts,
    title="Compatible Datasets by Processing Level",
    legend=True,
    top_n=3,
)

### Compatible Datasets by File Extension

Not all datasets have the `format` attribute, so we assess format using the `extension` attribute.


In [ ]:
extension_counts = compatible_datasets.extension.value_counts(dropna=False)
extension_names = extension_counts.index

plotting.create_pie_chart(
    labels=extension_names,
    sizes=extension_counts,
    title="Compatible Datasets by File Extension",
    legend=True,
)

## What datasets are incompatible or inconclusive, and why?

The 2026 assessment reports normalized categories derived from `assessment_status`, `failure_category`, `failure_subcategory`, `error_code`, HTTP status fields, endpoints, and selected `raw_error_body` patterns. Service errors and access failures are reported separately from proven data incompatibilities so operational issues do not distort the data-compatibility picture. In practice, inconclusive datasets should be treated as likely incompatible until a targeted follow-up confirms otherwise. The **Unsupported asset** category usually means the assessment found a granule data asset, but TiTiler-CMR could not select a supported reader for it because the asset has an unsupported or unrecognized file type, media type, file signature, or because the asset href/extension metadata needed to identify the file type was missing.



In [ ]:
category_counts = df["report_category"].value_counts()
plotting.create_pie_chart(
    labels=category_counts.index,
    sizes=category_counts,
    title="Assessment Results by Report Category",
    legend=True,
    top_n=7,
    legend_position="center left",
)

### Assessment status breakdown

`assessment_status` remains visible because an inconclusive result is not the same as a proven incompatible dataset. Most inconclusive rows are probably incompatible with TiTiler-CMR today, but they are also the rows most worth revisiting when the blocker is missing granules, missing or invalid bbox metadata, service errors, or sampling/probe limits.



In [ ]:
status_counts = df["assessment_status"].value_counts(dropna=False)
plotting.create_pie_chart(
    labels=status_counts.index,
    sizes=status_counts,
    title="Assessment Status Breakdown",
    legend=True,
)

### Inconclusive raw error summary

These are the most common raw error bodies among inconclusive operational outcomes. Metadata gaps are excluded here so service, render, request, and access errors are easier to inspect.


In [ ]:
operational_categories = [
    "Service error",
    "Render error",
    "Request error",
    "Inaccessible or authentication",
]
inconclusive_errors = df[
    df["assessment_status"].eq("inconclusive")
    & df["report_category"].isin(operational_categories)
].copy()
raw_errors = inconclusive_errors["raw_error_body"].fillna("").astype(str).str.strip()
inconclusive_errors["Raw Error Snippet"] = raw_errors.mask(
    raw_errors.eq(""), "No raw error body"
).str.replace("\n", " ", regex=False)
inconclusive_errors["Raw Error Snippet"] = inconclusive_errors[
    "Raw Error Snippet"
].str.slice(0, 180)

raw_error_summary = (
    inconclusive_errors.groupby(
        ["report_category", "report_reason_detail", "Raw Error Snippet"],
        dropna=False,
    )
    .size()
    .reset_index(name="Datasets")
    .sort_values("Datasets", ascending=False)
    .head(15)
    .rename(
        columns={
            "report_category": "Category",
            "report_reason_detail": "Reason Detail",
        }
    )
)
raw_error_summary

### Data incompatibilities

Data incompatibilities are cases where the granule asset had an acceptable file type, but the structure or contents of the data were not tileable by TiTiler-CMR. In other words, TiTiler-CMR could choose a reader for the file, but the opened dataset did not satisfy the assumptions needed to render map tiles. A basic requirement for TiTiler-CMR compatibility is gridded data with usable spatial coordinates, either X/Y or latitude/longitude coordinates. Many incompatible datasets do not meet that requirement. Common examples include missing X/Y or lat/lon spatial coordinate metadata, no tileable variables, unsupported dimensionality, unsupported file signatures, and decode errors.



In [ ]:
data_incompatible_subset_df = df[df["report_category"] == "Data incompatibility"]
data_reason_counts = data_incompatible_subset_df["report_reason"].value_counts(
    dropna=False
)
plotting.create_pie_chart(
    labels=data_reason_counts.index,
    sizes=data_reason_counts,
    title="Data Incompatibilities by Reason",
    legend=True,
    top_n=6,
    legend_position="center left",
)

### Metadata, access, and service outcomes

Metadata gaps include missing granules and invalid or missing granule bounding boxes. Inaccessible/authentication failures include S3 credential lookup failures, including IMDS token lookup failures observed in `raw_error_body`. Service, render, and request errors are kept as operational categories; they should not be treated as compatible just because they may be transient. They are better read as likely incompatible results that deserve targeted investigation if the dataset is important.

# Summary and next steps

The current report is regenerated from `assessment-results-2026-07-01.parquet` and uses shared categorization code in `reporting.py`. Individual datasets may still need targeted testing before being advertised as supported, but the normalized categories make future assessment updates easier to review.

Potential follow-up work:

- File endpoint issues for recurring server-side errors, such as `NoneType` failures returned as HTTP 500 responses.
- Re-run the assessment after endpoint or credential-environment fixes.
- Continue improving metadata-based identification of datasets that should work with TiTiler-CMR.

